# SmartBasket · Notebook 04 — First Solar agent on the A-Priori rules

**Team Frenchies** · Introduction to Big Data · Week 11

The user describes a basket in plain language. **Upstage Solar** decides which of our two tools to call, with which arguments, and writes the answer from what the tools return. The tools read the association rules mined by A-Priori on all 3.2 million orders (notebook 02). The model never invents a product or a number.

| Tool | What it does |
|---|---|
| `search_products(query)` | turns words like "avocado" into exact product names that exist in our rules |
| `recommend_for_basket(products)` | returns the products most associated with the basket, ranked by **lift**, with confidence and lift as the reason |

This covers the three project conditions at once: a course method (frequent itemsets), an LLM agent with tool calling, and the Upstage Solar model.

> 🇫🇷 *Pas besoin des données Instacart ici : le notebook lit directement le fichier de règles d'Alexandre sur GitHub. Il faut seulement une clé Upstage dans les Secrets Colab, sous le nom `UPSTAGE_API_KEY`.*

In [ ]:
# 🇫🇷 Installation et paramètres
!pip -q install openai

import os, json, re
import numpy as np
import pandas as pd

OUT = "out/agent_week11"
os.makedirs(OUT, exist_ok=True)

# Rules mined on all prior orders: 0.01 % support (notebook 03, 3,679 products) if it is on GitHub, else 0.05 % (notebook 02, 763 products)
BASE = "https://raw.githubusercontent.com/arthurcomet/smartbasket-frenchies/main/"
RULES_CANDIDATES = [os.environ.get("SB_RULES_PATH"),
                    BASE + "out/pcy_week11/rules_full_support_0p01.csv",
                    BASE + "out/apriori_20261118/rules_full_support_0p05.csv"]

BASE_URL = "https://api.upstage.ai/v1"    # if you get a 404, try "https://api.upstage.ai/v1/solar"
MODEL_CANDIDATES = ["solar-pro4", "solar-mini"]   # tried in order; solar-pro2 and solar-pro3 are deprecated (Nov 2026)

## 1 · The rules and the two tools (plain Python, no LLM yet)

In [ ]:
# 🇫🇷 Chargement des règles et index par produit
for RULES_PATH in [p for p in RULES_CANDIDATES if p]:
    try:
        rules = pd.read_csv(RULES_PATH)
        break
    except Exception as e:
        print(f"not available ({type(e).__name__}): {RULES_PATH}")
print("rules file:", RULES_PATH)
rules = rules.sort_values("lift", ascending=False)
NAME_OF = dict(zip(rules["a_id"], rules["if_bought"]))
NAME_OF.update(zip(rules["b_id"], rules["then_buy"]))
ID_OF = {name.lower(): pid for pid, name in NAME_OF.items()}
# how many baskets contain the product: support / confidence of any rule starting from it
BASKETS_WITH = (rules["support"] / rules["confidence"]).round().groupby(rules["a_id"]).first().to_dict()
BY_PRODUCT = {a: g for a, g in rules.groupby("a_id")}
print(f"{len(rules):,} rules, {len(NAME_OF):,} products appear in at least one rule")

In [ ]:
# 🇫🇷 Outil 1 : retrouver les noms exacts des produits à partir de mots simples
STOP = {"organic", "the", "of", "and", "with", "a", "fresh", "large", "small", "bag", "count"}

def _words(text):
    return [w for w in re.findall(r"[a-z0-9%]+", text.lower())]

def _stem(w):
    return w[:-1] if len(w) > 3 and w.endswith("s") else w

def search_products(query, limit=6):
    """Exact product names from our rules that match the words of the query."""
    q = [_stem(w) for w in _words(query) if w not in STOP] or [_stem(w) for w in _words(query)]
    scored = []
    for pid, name in NAME_OF.items():
        n = {_stem(w) for w in _words(name)}
        hits = sum(any(t == w or w.startswith(t) for w in n) for t in q)
        if hits:
            scored.append((hits, BASKETS_WITH.get(pid, 0), name))
    scored.sort(key=lambda x: (-x[0], -x[1]))
    best = [s for s in scored if s[0] == scored[0][0]] if scored else []
    return {"query": query,
            "matches": [{"product": name, "baskets": int(n)} for _, n, name in best[:limit]],
            "note": "" if best else "No product with these words in our rules (only products frequent enough are covered)."}

search_products("avocado")

In [ ]:
# 🇫🇷 Outil 2 : recommander à partir du panier, classé par lift (avec la raison chiffrée)
def _similar(a, b):
    """True if two product names share 2+ meaningful words, like two yogurt flavours (a substitute, not a complement)."""
    wa = {_stem(w) for w in _words(a)} - STOP
    wb = {_stem(w) for w in _words(b)} - STOP
    return len(wa & wb) >= 2

def recommend_for_basket(products, k=5, skip_similar=True):
    """Products most associated with the basket, ranked by lift."""
    basket, unknown = [], []
    for p in products:
        pid = ID_OF.get(p.lower())
        if pid is None:
            m = search_products(p)["matches"]
            pid = ID_OF.get(m[0]["product"].lower()) if m else None
        (basket.append(pid) if pid is not None else unknown.append(p))
    best = {}
    for a in basket:
        g = BY_PRODUCT.get(a)
        if g is None:
            continue
        for b, lift, conf in zip(g["b_id"], g["lift"], g["confidence"]):
            if b in basket or lift <= 1:
                continue
            if skip_similar and any(_similar(NAME_OF[x], NAME_OF[b]) for x in basket):
                continue
            if b not in best or lift > best[b][0]:
                best[b] = (lift, conf, a)
    top = sorted(best.items(), key=lambda x: -x[1][0])[:k]
    recs = [{"product": NAME_OF[b], "because_you_have": NAME_OF[a],
             "confidence": round(float(conf), 3), "lift": round(float(lift), 2),
             "explanation": f"{conf:.0%} of the baskets with {NAME_OF[a]} also contain {NAME_OF[b]}, "
                            f"{lift:.1f} times more often than an average basket."}
            for b, (lift, conf, a) in top]
    return {"basket": [NAME_OF[p] for p in basket], "not_found": unknown, "recommendations": recs,
            "note": "" if recs else "No association rule for these products."}

recommend_for_basket(["Limes", "Organic Hass Avocado"])

In [ ]:
# 🇫🇷 Pourquoi on écarte les produits « similaires » : sans filtre, on propose un autre parfum du même yaourt
yogurt = search_products("almond milk yogurt")["matches"][0]["product"]
print("Basket:", yogurt)
print("without filter:", [r["product"] for r in recommend_for_basket([yogurt], skip_similar=False)["recommendations"]])
print("with filter   :", [r["product"] for r in recommend_for_basket([yogurt])["recommendations"]])

## 2 · Connect Upstage Solar

Create a key on **console.upstage.ai**, then in Colab: 🔑 *Secrets* → add `UPSTAGE_API_KEY` → switch on *Notebook access*. Never paste the key in the code: the repository is public.

In [ ]:
# 🇫🇷 Connexion à Solar (la clé vient des Secrets Colab, jamais du code)
from openai import OpenAI
try:
    from google.colab import userdata
    API_KEY = userdata.get("UPSTAGE_API_KEY")
except Exception:
    API_KEY = os.environ.get("UPSTAGE_API_KEY")

client, MODEL = (OpenAI(api_key=API_KEY, base_url=BASE_URL) if API_KEY else None), None
if client is None:
    print("No UPSTAGE_API_KEY found: add it in Colab secrets, then run this cell again.")
for m in MODEL_CANDIDATES if client else []:
    try:
        r = client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say hello in five words."}])
        MODEL = m
        print(m, "answers:", r.choices[0].message.content)
        break
    except Exception as e:
        print(f"{m}: {type(e).__name__}: {str(e)[:160]}")
if client is not None and MODEL is None:
    client = None
    print("No model answered: check credits and the model list on console.upstage.ai/docs/models.")

In [ ]:
# 🇫🇷 Description des outils pour Solar (format « function calling »)
TOOLS = [
    {"type": "function", "function": {
        "name": "search_products",
        "description": "Find the exact product names in our store data that match simple words, e.g. 'avocado' or 'pasta sauce'. Call it before recommend_for_basket.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "words describing one product"}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "recommend_for_basket",
        "description": "Recommend products that are often bought together with the basket, from association rules mined with A-Priori on 3.2 million real grocery orders. Returns lift and confidence for each recommendation.",
        "parameters": {"type": "object", "properties": {
            "products": {"type": "array", "items": {"type": "string"},
                         "description": "exact product names, as returned by search_products"},
            "k": {"type": "integer", "description": "number of recommendations, default 5"}},
            "required": ["products"]}}},
]
TOOL_FUNCS = {"search_products": search_products, "recommend_for_basket": recommend_for_basket}

SYSTEM = """You are SmartBasket, a grocery shopping assistant.
1. First call search_products for each product the user mentions, to get exact product names. Product names are in English: translate the user's words to English before searching.
2. Then call recommend_for_basket with those exact names.
3. Only recommend products returned by recommend_for_basket. Never invent a product or a number.
4. Explain each recommendation with the numbers from the tool, in simple words.
5. If the tools find nothing, say so honestly.
Answer in the user's language, in 3 to 6 short lines."""

In [ ]:
# 🇫🇷 La boucle de l'agent : Solar demande un outil, on l'exécute, on lui renvoie le résultat, jusqu'à la réponse finale
def ask(user_message, max_steps=6, show=True):
    if client is None:
        raise RuntimeError("Connect Solar first (UPSTAGE_API_KEY).")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_message}]
    calls = []
    for _ in range(max_steps):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS,
                                             tool_choice="auto", temperature=0).choices[0].message
        if not msg.tool_calls:
            if show:
                print(f"\n🧑 {user_message}\n🤖 {msg.content}")
            return {"user": user_message, "tool_calls": calls, "answer": msg.content}
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_FUNCS[tc.function.name](**args)
            calls.append({"tool": tc.function.name, "arguments": args, "result": result})
            if show:
                print(f"🔧 {tc.function.name}({args})")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return {"user": user_message, "tool_calls": calls, "answer": None, "error": "too many steps"}

## 3 · Five test conversations

The last one is on purpose: a laptop and a phone charger are not groceries, so the agent must say it has nothing, not invent something.

In [ ]:
# 🇫🇷 Cinq conversations de test, sauvegardées comme preuve pour le rapport
TESTS = [
    "I'm making guacamole, I already have avocados and limes. What else should I buy?",
    "Pasta night: I have spaghetti and marinara sauce in my cart.",
    "Tacos tonight with ground beef and taco seasoning, what am I missing?",
    "J'ai des oeufs et du pain dans mon panier, tu me conseilles quoi ?",
    "I need a phone charger and a new laptop.",
]
transcripts = []
for t in TESTS:
    try:
        transcripts.append(ask(t))
    except Exception as e:
        print(f"\n🧑 {t}\n⚠️ {type(e).__name__}: {e}")
        transcripts.append({"user": t, "error": f"{type(e).__name__}: {e}"})

with open(f"{OUT}/transcripts.json", "w") as f:
    json.dump(transcripts, f, indent=2, ensure_ascii=False, default=str)
print(f"\nsaved {OUT}/transcripts.json")

In [ ]:
# 🇫🇷 Bilan chiffré : combien de conversations ont utilisé les deux outils, et combien ont planté
ok = [t for t in transcripts if t.get("answer")]
summary = {
    "model": MODEL,
    "rules_file": RULES_PATH,
    "conversations": len(transcripts),
    "answered": len(ok),
    "used_search_then_recommend": sum(1 for t in ok if [c["tool"] for c in t["tool_calls"]][:1] == ["search_products"]
                                      and any(c["tool"] == "recommend_for_basket" for c in t["tool_calls"])),
    "tool_calls_total": sum(len(t.get("tool_calls", [])) for t in transcripts),
    "errors": [t["error"] for t in transcripts if "error" in t],
}
with open(f"{OUT}/evidence_week11_agent.json", "w") as f:
    json.dump(summary, f, indent=2)
summary

## 4 · What to say at the presentation

> *"The agent runs on Upstage Solar with two tools. The first finds the exact product names from simple words, the second reads our A-Priori rules and ranks recommendations by lift. Solar decides itself which tool to call. For tacos, it [answer]. When we ask for a laptop, it says it found nothing: it does not invent anything."*

> 🇫🇷 *Problème trouvé et corrigé : les meilleures règles par lift étaient souvent un autre parfum du même produit (yaourt myrtille → yaourt pêche). Ce sont des substituts, pas des produits complémentaires. On les écarte pour l'instant avec un filtre simple sur les noms, et en semaine 12 on fera ça proprement avec MinHash/LSH (cours semaine 3).*

In [ ]:
# 🇫🇷 Télécharger tous les résultats en un zip (à mettre ensuite sur GitHub, dans le dossier out/)
import shutil
zip_path = shutil.make_archive(OUT.rstrip("/").split("/")[-1], "zip", root_dir="out", base_dir=OUT.rstrip("/").split("/")[-1])
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("created", zip_path)